# %% [markdown]
# # AdaptiveSRE GRPO Training (Colab)
# 
# **Hardware required:** T4 GPU (16GB VRAM)
# 
# **Runtime → Change runtime type → GPU**

# %% [markdown]
# ## 1. Install Dependencies

# %%
!pip install -q unsloth trl transformers torch httpx fastapi pydantic
!pip install -q openenv-core  # if available

# %% [markdown]
# ## 2. Clone Repo & Start Services

# %%
!git clone https://github.com/ashifsekh/adaptive-sre.git
%cd adaptive-sre

# Start mock services in background
!docker-compose -f mock_services/docker-compose.yml up -d

# Start server in background
!nohup python3 -m uvicorn server.app:app --port 8000 > server.log 2>&1 &

# Verify
import time
time.sleep(5)
!curl -s http://localhost:8000/health

# %% [markdown]
# ## 3. Run Training

# %%
!python train.py \
    --episodes 200 \
    --task hard \
    --output ./checkpoints/gen1/ \
    --batch_size 4 \
    --learning_rate 5e-6 \
    --save_every 50 \
    --env_url http://localhost:8000

# %% [markdown]
# ## 4. Evaluate

# %%
!python eval.py \
    --trained_model ./checkpoints/gen1/final \
    --env_url http://localhost:8000 \
    --episodes 20 \
    --output eval_results.json

# %% [markdown]
# ## 5. Generate Plot

# %%
!python plot_rewards.py

from IPython.display import Image
Image("rewards_curve.png")